# Workday Sales RAG Model Deployment & Serving

This notebook handles model serving endpoint creation, workload selection, authentication configuration (Service Principal OAuth + PAT), readiness checks, endpoint testing, and deployment best practices for higher environments.

In [0]:
# catalog and schema parameters
dbutils.widgets.text("catalog_name", "rag_agentic")
dbutils.widgets.text("schema_name", "workday_demos")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name    = dbutils.widgets.get("schema_name")

# Model config parameters
dbutils.widgets.text("model_name", "workday_sales_rag")
dbutils.widgets.text("model_version", "")  # Leave empty to auto-deploy latest version
dbutils.widgets.text("llm_endpoint_name", "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.text("vs_endpoint_name", "sales-endpoint-rag_agentic")
dbutils.widgets.text("customer_feedback_index", "customer_feedback_index")
dbutils.widgets.text("meeting_notes_index", "meeting_notes_index")
dbutils.widgets.text("email_communications_index", "email_communications_index")
dbutils.widgets.text("num_results", "5")
dbutils.widgets.text("temperature", "0.1")
dbutils.widgets.text("max_tokens", "500")

model_name                  = dbutils.widgets.get("model_name")
model_version               = dbutils.widgets.get("model_version")
llm_endpoint_name           = dbutils.widgets.get("llm_endpoint_name")
vs_endpoint_name            = dbutils.widgets.get("vs_endpoint_name")
customer_feedback_index     = dbutils.widgets.get("customer_feedback_index")
meeting_notes_index         = dbutils.widgets.get("meeting_notes_index")
email_communications_index  = dbutils.widgets.get("email_communications_index")
num_results                 = dbutils.widgets.get("num_results")
temperature                 = dbutils.widgets.get("temperature")
max_tokens                  = dbutils.widgets.get("max_tokens")

# Compute parameters
dbutils.widgets.text("workload_type", "CPU")
dbutils.widgets.text("workload_size", "Small")
dbutils.widgets.text("scale_to_zero_enabled", "true")

workload_type               = dbutils.widgets.get("workload_type")
workload_size               = dbutils.widgets.get("workload_size")
scale_to_zero_enabled       = dbutils.widgets.get("scale_to_zero_enabled").lower() == "true"

# serving endpoint and secret parameters
dbutils.widgets.text("secret_scope", "agentic")
dbutils.widgets.text("serving_endpoint_name", "workday_sales_rag_endpoint")

secret_scope                = dbutils.widgets.get("secret_scope")
serving_endpoint_name       = dbutils.widgets.get("serving_endpoint_name")

In [0]:
import mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk.errors import NotFound
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput, ServingModelWorkloadType

In [0]:
def needs_update(current_entity, desired_entity) -> tuple[bool, list[str]]:
    """Check if endpoint configuration needs updating."""
    changes = []
    
    # Check model version
    if current_entity.entity_version != desired_entity.entity_version:
        changes.append(f"Version: {current_entity.entity_version} → {desired_entity.entity_version}")
    
    # Check workload configuration
    if current_entity.workload_size != desired_entity.workload_size:
        changes.append(f"Workload Size: {current_entity.workload_size} → {desired_entity.workload_size}")
    
    if current_entity.scale_to_zero_enabled != desired_entity.scale_to_zero_enabled:
        changes.append(f"Scale to Zero: {current_entity.scale_to_zero_enabled} → {desired_entity.scale_to_zero_enabled}")
    
    # Check environment variables (key comparison only, not secret values)
    current_env_keys = set(current_entity.environment_vars.keys()) if current_entity.environment_vars else set()
    desired_env_keys = set(desired_entity.environment_vars.keys())
    
    if current_env_keys != desired_env_keys:
        missing = desired_env_keys - current_env_keys
        extra = current_env_keys - desired_env_keys
        if missing:
            changes.append(f"Missing env vars: {missing}")
        if extra:
            changes.append(f"Extra env vars: {extra}")
    
    return len(changes) > 0, changes


In [0]:
# Check if endpoint already exists
try:
    # Initialize Databricks SDK client
    w = WorkspaceClient()

    # Resolve model version: use provided version, or fetch latest from registry
    full_model_name = f"{catalog_name}.{schema_name}.{model_name}"
    
    if model_version and model_version.strip():
        model_version = model_version.strip()
        print(f"Using provided model version: {model_version}")
    else:
        mlflow_client = MlflowClient()
        all_versions = mlflow_client.search_model_versions(f"name='{full_model_name}'")
        if not all_versions:
            raise ValueError(f"No versions found for model '{full_model_name}'")
        model_version = str(max(int(v.version) for v in all_versions))
        print(f"No version specified — using latest model version: {model_version}")
    
    print(f"\nDeploying model: {full_model_name} (version {model_version})")

    # Endpoint configuration
    endpoint_name = serving_endpoint_name

    desired_env_vars = {
                            "DATABRICKS_HOST": w.config.host,
                            "DATABRICKS_CLIENT_ID": f"{{{{secrets/{secret_scope}/sp_client_id}}}}",
                            "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{secret_scope}/sp_client_secret}}}}",
                            "LLM_ENDPOINT_NAME": llm_endpoint_name,
                            "VS_ENDPOINT_NAME": vs_endpoint_name,
                            "CUSTOMER_FEEDBACK_INDEX": f"{catalog_name}.{schema_name}.{customer_feedback_index}",
                            "MEETING_NOTES_INDEX": f"{catalog_name}.{schema_name}.{meeting_notes_index}",
                            "EMAIL_COMMUNICATIONS_INDEX": f"{catalog_name}.{schema_name}.{email_communications_index}",
                            "NUM_RESULTS": num_results,
                            "TEMPERATURE": temperature,
                            "MAX_TOKENS": max_tokens
                        }

    desired_entity = ServedEntityInput(
                                        entity_name=full_model_name,
                                        entity_version=str(model_version),
                                        workload_type=ServingModelWorkloadType(workload_type),
                                        workload_size=workload_size,
                                        scale_to_zero_enabled=scale_to_zero_enabled,
                                        environment_vars=desired_env_vars,
                                        )
    
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"\n Endpoint '{endpoint_name}' exists. Checking configuration...")
    
    # Get current configuration
    current_entities = list(existing_endpoint.config.served_entities if existing_endpoint.config and existing_endpoint.config.served_entities else [])
    
    if len(current_entities) == 0:
        raise ValueError(f"Endpoint exists but has no served entities")
    
    current_entity = current_entities[0]
    
    # Validate model name matches
    if current_entity.entity_name != full_model_name:
        print(f"\n ERROR: Endpoint currently serves: {current_entity.entity_name} \n Cannot replace with: {full_model_name} \n Choose a different endpoint name or delete the existing endpoint.")
        raise ValueError(f"Endpoint name collision: '{endpoint_name}' already serves '{current_entity.entity_name}'")
    
    # Check if update is needed (idempotency check)
    update_needed, changes = needs_update(current_entity, desired_entity)
    
    if not update_needed:
        print(f"\n Endpoint configuration is already up-to-date! \n Model: {model_name} \n Version: {model_version} \n Workload: {workload_type} / {workload_size} \n Scale to Zero: {scale_to_zero_enabled} \n  No changes needed. Endpoint is ready to use.")
        action_taken = "no_change"
    else:
        print(f"\n Configuration changes detected:")
        for change in changes:
            print(f"{change}")
        
        print(f"\n Updating endpoint configuration...")
        w.serving_endpoints.update_config(name=endpoint_name, served_entities=[desired_entity])
        print(f"\n Endpoint update initiated!")
        action_taken = "updated"
    
except NotFound:
    # Create new endpoint
    print(f"\n Endpoint '{endpoint_name}' does not exist. Creating...")

    endpoint_config = EndpointCoreConfigInput(served_entities=[desired_entity])
    w.serving_endpoints.create(name=endpoint_name, config=endpoint_config)

    print(f"\n Endpoint creation initiated!")
    action_taken = "created"

except Exception as e:
    print(f"\n Error during endpoint deployment: {str(e)}")
    raise

# Summary
print(f"\n" + "=" * 60)
print(f" Deployment Summary")
print("=" * 60)
print(f"   Endpoint: {endpoint_name}")
print(f"   Action: {action_taken.upper().replace('_', ' ')}")
print(f"   Model: {model_name} v{model_version}")
print(f"\n Endpoint URL:")
print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")

if action_taken in ["created", "updated"]:
    print(f"\n Proceeding to readiness check...")
else:
    print(f"\n Endpoint is ready to use (no deployment needed)")